- Problem: LLMs tend to hallucinate and give black box reasoning
    - R-Retrieval
    - A-Augmentation
    - G-Generation

- Retrieve relevant information from querying external sources —> Augment with pre-trained knowledge —> Generate response
 
- Data indexing
- Extract text from documents and transform them into vector embeddings. This preserves semantic meaning by assigning similar embedding values to co-related words. Suppose the words king and queen are correlated, so if the word king has a vector embedding [0.15 0.55  0.95], then the word queen would have an embedding that is very close to it, like [0.13  0.57  0.94]. It forms our retrieval Database.

- Data Retrieval and Generation
- The user's query is transformed into an embedding using the same model as earlier, and now Euclidean distance is calculated with the vectors in the database, and the closest ones are chosen to augment with the pretrained knowledge to generate the response.

In [ ]:
import os
import faiss
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# -----------------------------
# 1. Chunking
# -----------------------------
def chunk_text(text, size=200, overlap=50):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

In [ ]:
# -----------------------------
# 2. Load documents
# -----------------------------
def load_docs():
    raw = """
    RAG stands for Retrieval Augmented Generation.
    It improves LLM accuracy using external data.
    Embeddings map text to vectors.
    FAISS enables fast similarity search.
    Hybrid search combines keyword + semantic retrieval.
    """
    return chunk_text(raw)

In [ ]:
# -----------------------------
# 3. Embeddings
# -----------------------------
def embed(texts):
    res = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([d.embedding for d in res.data]).astype("float32")

In [ ]:
# -----------------------------
# 4. Vector Store (FAISS)
# -----------------------------
class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)
        self.texts = []

    def add(self, texts, vectors):
        self.texts.extend(texts)
        faiss.normalize_L2(vectors)
        self.index.add(vectors)

    def search(self, query_vec, k=3):
        faiss.normalize_L2(query_vec)
        scores, indices = self.index.search(query_vec, k)
        return [self.texts[i] for i in indices[0]]

In [ ]:
# -----------------------------
# 5. Re-ranking (simple)
# -----------------------------
def rerank(query, docs):
    scored = [(doc, len(set(query.split()) & set(doc.split()))) for doc in docs]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scored]

In [ ]:
# -----------------------------
# 6. RAG pipeline
# -----------------------------
def rag(query, store):
    q_vec = embed([query])
    retrieved = store.search(q_vec)

    reranked = rerank(query, retrieved)
    context = "\n".join(reranked[:3])

    prompt = f"""
    You are a helpful assistant.
    Answer ONLY from the context below.
    If unsure, say "I don't know".

    Context:
    {context}

    Question: {query}
    """

    res = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content

In [ ]:
# -----------------------------
# 7. Build pipeline
# -----------------------------
docs = load_docs()
vectors = embed(docs)

store = VectorStore(vectors.shape[1])
store.add(docs, vectors)


In [ ]:
# -----------------------------
# 8. Run loop
# -----------------------------
while True:
    q = input("Ask: ")
    if q.lower() == "exit":
        break
    print(rag(q, store))